In [6]:
# import library
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
# setting
CSV_PATH = "/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv"
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 512


In [7]:
# load data
df = pd.read_csv(CSV_PATH)
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [8]:
# simple EDA
print(df["sentiment"].value_counts(normalize=True))

lengths = df["review"].str.split().apply(len)
print(lengths.describe(include="all"))

print(np.percentile(lengths, 90))

'''
for t in df['review'].sample(5): 
    print(t, "\n---")
'''

sentiment
positive    0.5
negative    0.5
Name: proportion, dtype: float64
count    50000.000000
mean       231.156940
std        171.343997
min          4.000000
25%        126.000000
50%        173.000000
75%        280.000000
max       2470.000000
Name: review, dtype: float64
451.0


'\nfor t in df[\'review\'].sample(5): \n    print(t, "\n---")\n'

In [9]:
class IMDBDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }


In [10]:
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

df_train_val, df_test = train_test_split(
    df, 
    test_size=0.10, 
    random_state=42, 
    stratify=df['sentiment']
)


df_train, df_val = train_test_split(
    df_train_val, 
    test_size=0.2222, 
    random_state=42, 
    stratify=df_train_val['sentiment']
)

df_test.to_csv("holdout_test_set.csv", index=False)

# Reset indices
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = IMDBDataset(df_train['review'], df_train['sentiment'], tokenizer, MAX_LENGTH)
val_dataset = IMDBDataset(df_val['review'], df_val['sentiment'], tokenizer, MAX_LENGTH)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=10,
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()
model.save_pretrained("./fine_tuned_distilbert")
tokenizer.save_pretrained("./fine_tuned_distilbert")
print("Model and tokenizer saved successfully to './fine_tuned_distilbert'")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, 

Epoch,Training Loss,Validation Loss
1,0.509448,0.483989
2,0.345390,0.510611
3,0.155801,0.637221


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved successfully to './fine_tuned_distilbert'


In [11]:
def get_prediction(text, tokenizer, model, device):
    """Helper function to run inference on a single string."""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    
    # Move batch inputs to the designated device (GPU or CPU)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probabilities = F.softmax(logits, dim=1).flatten()
        prediction = torch.argmax(logits, dim=1).item()
        
    class_mapping = {0: "Negative", 1: "Positive"}
    return class_mapping[prediction], probabilities[prediction].item()


def main():
    MODEL_PATH = "./fine_tuned_distilbert"
    
    # 1. Setup Device (Utilize GPU if available)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running inference execution on device: {device.type.upper()}")

    # 2. Load fine-tuned assets
    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(device)
        model.eval()
    except FileNotFoundError:
        print(f"Error: Model files not found at '{MODEL_PATH}'. Please run train.py first.")
        return

    # =========================================================================
    # Test Scenario A: Ad-hoc manual text strings
    # =========================================================================
    print("\n=== Scenario A: Testing Custom Inputs ===")
    sample_phrases = [
        "This tool completely streamlined our system pipeline, incredible!",
        "Absolute garbage. Broke instantly and zero support options."
    ]
    
    for phrase in sample_phrases:
        label, conf = get_prediction(phrase, tokenizer, model, device)
        print(f"Input: '{phrase}'\nPrediction: {label} ({conf*100:.2f}% confidence)\n")

    # =========================================================================
    # Test Scenario B: Evaluating the 10% Holdout Data Split
    # =========================================================================
    print("=== Scenario B: Testing Unseen Holdout File ===")
    try:
        test_df = pd.read_csv("holdout_test_set.csv")
        
        # Test the first 5 entries from the 10% holdout partition
        for idx, row in test_df.head(5).iterrows():
            label, conf = get_prediction(row['review'], tokenizer, model, device)
            # Map original true labels back for evaluation visualization
            true_label = "Positive" if row['sentiment'] == 1 else "Negative"
            
            print(f"Text: '{row['review']}'")
            print(f"Predicted: {label} ({conf*100:.2f}%) | Actual: {true_label}")
            print("-" * 50)
            
    except FileNotFoundError:
        print("Note: 'holdout_test_set.csv' not found. Skipping dataset verification.")

if __name__ == "__main__":
    main()

Running inference execution on device: CUDA


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


=== Scenario A: Testing Custom Inputs ===
Input: 'This tool completely streamlined our system pipeline, incredible!'
Prediction: Positive (96.07% confidence)

Input: 'Absolute garbage. Broke instantly and zero support options.'
Prediction: Negative (99.12% confidence)

=== Scenario B: Testing Unseen Holdout File ===
Text: 'I'm not kidding about that summary and vote! The video distributors have packaged this as just another typical '80s werewolf movie, but it is in fact the greatest parody of the horror genre that you can imagine, having done for the horror movie what "Blazing Saddles" did for the western. I have seen plenty of comedies - good, bad, stupid, weird, etc. (usually walking away unimpressed), and I think that comedy must be the most difficult genre for filmmakers and actors to work in - it takes just the right kind of touch to make things successful, and part of that is having good ideas. "Full Moon High" is bulging with good ideas - so many, in fact, that it can easily pu